In [17]:
"""
Hospital Resource Utilization & Efficiency Analysis
Author: Carolina Moreira
Dataset: Synthetic hospital operations dataset (hospital beds management)
Purpose: Analyze patient flow, bed utilization, staffing efficiency, and
         service performance to identify operational improvement opportunities.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── 0. Style ──────────────────────────────────────────────────────────────────
PALETTE  = ["#1F4E79", "#2E75B6", "#70AD47", "#ED7D31", "#A9D18E"]
ACCENT   = "#1F4E79"
LIGHT    = "#BDD7EE"
sns.set_theme(style="whitegrid", palette=PALETTE)
plt.rcParams.update({"font.family": "sans-serif", "axes.spines.top": False,
                     "axes.spines.right": False})

OUT = "figures/"

# ── 1. Load data ──────────────────────────────────────────────────────────────
staff     = pd.read_csv("/content/drive/MyDrive/AML proj/staff.csv")
schedule  = pd.read_csv("/content/drive/MyDrive/AML proj/staff_schedule.csv")
patients  = pd.read_csv("/content/drive/MyDrive/AML proj/patients.csv")
services  = pd.read_csv("/content/drive/MyDrive/AML proj/services_weekly.csv")

In [18]:
# ── 2. Feature engineering ────────────────────────────────────────────────────
# Bed utilization rate = admitted / available
services["utilization_rate"] = services["patients_admitted"] / services["available_beds"]

# Demand pressure = requests / available beds  (how overloaded is each service?)
services["demand_pressure"] = services["patients_request"] / services["available_beds"]

# Refusal rate = refused / requested
services["refusal_rate"] = services["patients_refused"] / services["patients_request"]

# Patient length of stay
patients["arrival_date"]    = pd.to_datetime(patients["arrival_date"])
patients["departure_date"]  = pd.to_datetime(patients["departure_date"])
patients["los_days"]        = (patients["departure_date"] - patients["arrival_date"]).dt.days

# Staff attendance rate per week
schedule["present"] = schedule["present"].astype(int)
attendance = (schedule.groupby(["week", "service"])["present"]
              .mean().reset_index().rename(columns={"present": "attendance_rate"}))

services_att = services.merge(attendance, on=["week", "service"], how="left")

SERVICE_LABELS = {
    "emergency": "Emergency",
    "surgery": "Surgery",
    "general_medicine": "General Medicine",
    "ICU": "ICU"
}
services["service_label"] = services["service"].map(SERVICE_LABELS)
patients["service_label"]  = patients["service"].map(SERVICE_LABELS)

print("Data loaded. Generating figures...\n")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Bed utilization rate by service (weekly average)
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))

util_by_service = (services.groupby("service_label")["utilization_rate"]
                   .mean().sort_values(ascending=False).reset_index())

bars = ax.barh(util_by_service["service_label"], util_by_service["utilization_rate"],
               color=PALETTE[:4], edgecolor="white", height=0.55)

ax.axvline(1.0, color="red", linewidth=1.4, linestyle="--", label="100% capacity")
ax.axvline(0.85, color="#ED7D31", linewidth=1.2, linestyle=":", label="85% efficiency target")

for bar, val in zip(bars, util_by_service["utilization_rate"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.0%}", va="center", fontsize=10, fontweight="bold", color=ACCENT)

ax.set_xlabel("Average Bed Utilization Rate", fontsize=11)
ax.set_title("Bed Utilization Rate by Service\n(Average Across 52 Weeks)",
             fontsize=13, fontweight="bold", color=ACCENT, pad=12)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUT + "fig1_bed_utilization.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Figure 1 saved")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Demand pressure vs. refusal rate (4-quadrant scatter)
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 6))

svc_summary = services.groupby("service_label").agg(
    demand_pressure=("demand_pressure", "mean"),
    refusal_rate=("refusal_rate", "mean"),
    utilization_rate=("utilization_rate", "mean")
).reset_index()

scatter = ax.scatter(svc_summary["demand_pressure"], svc_summary["refusal_rate"],
                     s=svc_summary["utilization_rate"] * 800,
                     c=PALETTE[:4], edgecolors="white", linewidth=1.5, zorder=3)

for _, row in svc_summary.iterrows():
    ax.annotate(row["service_label"],
                xy=(row["demand_pressure"], row["refusal_rate"]),
                xytext=(8, 6), textcoords="offset points",
                fontsize=10, fontweight="bold", color=ACCENT)

med_dp = svc_summary["demand_pressure"].median()
med_rr = svc_summary["refusal_rate"].median()
ax.axvline(med_dp, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
ax.axhline(med_rr, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)

ax.set_xlabel("Demand Pressure (Requests / Available Beds)", fontsize=11)
ax.set_ylabel("Patient Refusal Rate", fontsize=11)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title("Demand Pressure vs. Refusal Rate by Service\n(Bubble size = avg bed utilization)",
             fontsize=13, fontweight="bold", color=ACCENT, pad=12)
plt.tight_layout()
plt.savefig(OUT + "fig2_demand_vs_refusal.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Figure 2 saved")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Weekly refusal rate trends by service (line chart)
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5))

for i, (svc, grp) in enumerate(services.groupby("service_label")):
    grp_sorted = grp.sort_values("week")
    ax.plot(grp_sorted["week"], grp_sorted["refusal_rate"],
            label=svc, color=PALETTE[i], linewidth=1.8, alpha=0.85)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xlabel("Week of Year", fontsize=11)
ax.set_ylabel("Patient Refusal Rate", fontsize=11)
ax.set_title("Patient Refusal Rate Over Time by Service\n(Weeks 1–52)",
             fontsize=13, fontweight="bold", color=ACCENT, pad=12)
ax.legend(title="Service", fontsize=9)
plt.tight_layout()
plt.savefig(OUT + "fig3_refusal_trends.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Figure 3 saved")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Staff attendance vs. patient satisfaction (by service)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)

for i, (svc, grp) in enumerate(services_att.groupby("service")):
    label = SERVICE_LABELS.get(svc, svc)
    grp = grp.dropna(subset=["attendance_rate"])
    axes[i].scatter(grp["attendance_rate"], grp["patient_satisfaction"],
                    color=PALETTE[i], alpha=0.6, s=30, edgecolors="white")
    # regression line
    m, b = np.polyfit(grp["attendance_rate"], grp["patient_satisfaction"], 1)
    x_range = np.linspace(grp["attendance_rate"].min(), grp["attendance_rate"].max(), 100)
    axes[i].plot(x_range, m * x_range + b, color=ACCENT, linewidth=1.5, linestyle="--")
    axes[i].set_title(label, fontsize=10, fontweight="bold", color=ACCENT)
    axes[i].set_xlabel("Attendance Rate", fontsize=9)
    if i == 0:
        axes[i].set_ylabel("Patient Satisfaction", fontsize=9)
    axes[i].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

fig.suptitle("Staff Attendance Rate vs. Patient Satisfaction by Service",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.savefig(OUT + "fig4_attendance_vs_satisfaction.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Figure 4 saved")

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 5 — Length of stay distribution by service
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))

los_clean = patients[patients["los_days"] >= 0]
order = (los_clean.groupby("service_label")["los_days"]
         .median().sort_values(ascending=False).index.tolist())

sns.boxplot(data=los_clean, x="service_label", y="los_days",
            order=order, palette=PALETTE[:4], ax=ax,
            width=0.5, flierprops={"marker": "o", "markersize": 3, "alpha": 0.4})

ax.set_xlabel("Service", fontsize=11)
ax.set_ylabel("Length of Stay (Days)", fontsize=11)
ax.set_title("Patient Length of Stay Distribution by Service",
             fontsize=13, fontweight="bold", color=ACCENT, pad=12)
plt.tight_layout()
plt.savefig(OUT + "fig5_length_of_stay.png", dpi=150, bbox_inches="tight")
plt.close()
print("✓ Figure 5 saved")

# ═══════════════════════════════════════════════════════════════════════════════
# SUMMARY STATS — printed to console (for README)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── KEY FINDINGS ─────────────────────────────────────────────────────")
for _, row in svc_summary.iterrows():
    print(f"{row['service_label']:20s} | Utilization: {row['utilization_rate']:.0%} "
          f"| Demand Pressure: {row['demand_pressure']:.1f}x "
          f"| Refusal Rate: {row['refusal_rate']:.0%}")

print(f"\nOverall avg refusal rate : {services['refusal_rate'].mean():.0%}")
print(f"Overall avg utilization  : {services['utilization_rate'].mean():.0%}")
print(f"Median LOS (all services): {los_clean['los_days'].median():.0f} days")
print(f"Highest demand pressure  : {svc_summary.loc[svc_summary['demand_pressure'].idxmax(), 'service_label']}")
print("─────────────────────────────────────────────────────────────────────")
print("\nAll figures saved to figures/")

import os
os.makedirs("figures", exist_ok=True)

Data loaded. Generating figures...

✓ Figure 1 saved
✓ Figure 2 saved
✓ Figure 3 saved
✓ Figure 4 saved
✓ Figure 5 saved

── KEY FINDINGS ─────────────────────────────────────────────────────
Emergency            | Utilization: 100% | Demand Pressure: 5.2x | Refusal Rate: 77%
General Medicine     | Utilization: 97% | Demand Pressure: 1.8x | Refusal Rate: 35%
ICU                  | Utilization: 84% | Demand Pressure: 1.0x | Refusal Rate: 12%
Surgery              | Utilization: 88% | Demand Pressure: 1.2x | Refusal Rate: 17%

Overall avg refusal rate : 35%
Overall avg utilization  : 92%
Median LOS (all services): 7 days
Highest demand pressure  : Emergency
─────────────────────────────────────────────────────────────────────

All figures saved to figures/
